In [16]:
import pandas as pd
import numpy as np

# --- 1. File Loading (Adjust this path as needed in Colab) ---
file_path = "/content/drive/MyDrive/25-11-07_Ref004_Edits.csv"
try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}. Total rows: {len(df)}")
except FileNotFoundError:
    print(f"Error: File not found at '{file_path}'. Please upload the file or check the path.")
    # In a real Colab notebook, you'd stop here if the file didn't load.
    pass

# --- 2. Pre-computation for Condition 4 ---
# We must first find all Author_Core_n values that meet the "conflict" criteria.
# Conflict criteria now include:
#   1. 1-2 characters long AND its paired Author_Role_n is EMPTY.
#   2. 1-2 characters long AND its paired Author_Role_n is NOT EMPTY,
#      BUT the same Author_Core_n appears with DIFFERENT non-empty Author_Role_n values elsewhere.

print("\nBuilding conflict author set (for Condition 4)...")
conflict_authors = set()

# --- Condition 1: Short author core AND empty role (matching previous logic) ---
conflict_cond1_authors = set()
for n in range(1, 8):
    core_col = f"Author_Core_{n}"
    role_col = f"Author_Role_{n}"
    role_is_empty = df[role_col].isna() | (df[role_col].astype(str).str.strip() == '')
    core_is_valid_len = df[core_col].notna() & (df[core_col].astype(str).str.len().isin([1, 2]))
    combined_mask = role_is_empty & core_is_valid_len
    authors_to_add = df.loc[combined_mask, core_col].astype(str).unique()
    conflict_cond1_authors.update(authors_to_add)

conflict_authors.update(conflict_cond1_authors)
print(f"Number of unique authors satisfying Condition 1 (short core, empty role): {len(conflict_cond1_authors)}")


# --- Condition 2: Short author core and multiple non-empty roles ---
# Collect all Author_Core and Author_Role pairs across all columns
all_author_data = []
for n in range(1, 8):
    temp_df = df[[f"Author_Core_{n}", f"Author_Role_{n}"]].copy()
    temp_df.columns = ["Author_Core", "Author_Role"]
    all_author_data.append(temp_df)

all_authors_df = pd.concat(all_author_data, ignore_index=True)

# Clean up whitespace and handle NaNs for consistent grouping
all_authors_df["Author_Core"] = all_authors_df["Author_Core"].astype(str).str.strip()
all_authors_df["Author_Role"] = all_authors_df["Author_Role"].astype(str).str.strip()

# Replace empty strings with NaN for easier handling of "empty" roles
all_authors_df["Author_Role"] = all_authors_df["Author_Role"].replace('', np.nan)

# Filter for authors with 1-2 characters
short_authors_df = all_authors_df[
    all_authors_df["Author_Core"].str.len().isin([1, 2])
].copy()

# Group by Author_Core and check the number of unique non-empty roles
role_counts = short_authors_df[short_authors_df["Author_Role"].notna()].groupby("Author_Core")["Author_Role"].nunique()

# Identify authors with more than one unique non-empty role
authors_with_multiple_roles = role_counts[role_counts > 1].index.tolist()

# Add these authors to the conflict set
conflict_authors.update(authors_with_multiple_roles)
print(f"Number of unique authors satisfying Condition 2 (short core, multiple non-empty roles): {len(authors_with_multiple_roles)}")


print(f"\nFound {len(conflict_authors)} unique conflict authors based on both criteria.")

# Print the first 30 authors in the conflict_authors set
print("\nFirst 30 conflict authors:")
print(list(conflict_authors)[:30])


# --- 3. Define the Flagging Function (Applies all 4 conditions) ---

def check_row_for_flag(row, conflict_set):
    """
    Checks a single row. Collects all column numbers (1-7) where all 4 conditions
    are met. Returns a formatted string "Y=col1,col2,..." or "N".
    """
    matching_columns = [] # List to store the column numbers that meet the conditions

    # Check each pair (n=1 to 7)
    for n in range(1, 8):
        author_core = row[f"Author_Core_{n}"]
        author_role = row[f"Author_Role_{n}"]

        # --- Start checking conditions ---

        # Skip if core author is blank/NaN, it can't match
        if pd.isna(author_core):
            continue

        author_core_str = str(author_core)

        # Condition 1: Author_Core_n is one or two characters long
        cond1 = len(author_core_str) in [1, 2]

        # Condition 2: Author_Role_n is NOT EMPTY
        cond2 = pd.notna(author_role) and str(author_role).strip() != ""

        # Condition 3: Author_Core_n does not end with "等"
        cond3 = not author_core_str.endswith("等")

        # Condition 4: Author_Core_n IS NOT in the conflict set
        cond4 = author_core_str not in conflict_set

        # --- Check all 4 ---
        if cond1 and cond2 and cond3 and cond4:
            # Found a match in this column. Add the column number to the list.
            matching_columns.append(str(n)) # Store as string for easy joining

    # After checking all 7 columns, determine the return value
    if matching_columns:
        # If any columns matched, return "Y=" followed by joined column numbers
        return "Y=" + ",".join(matching_columns)
    else:
        # If no columns matched, return "N"
        return 'N'

# --- 4. Apply the Function to Create the "Flag" Column ---

print("\nApplying flagging logic to all rows... (this may take a moment)")
df["Flag"] = df.apply(
    lambda row: check_row_for_flag(row, conflict_authors),
    axis=1
)
print("Flagging complete.")

# --- 5. Display Results ---
flag_summary = df["Flag"].value_counts()
print("\n--- Flag Summary ---")
print(flag_summary)

# Display a few rows that were flagged (where Flag is not 'N')
print("\nSample rows where 'Flag' is not 'N':")
flagged_rows = df[df["Flag"] != 'N']
print(flagged_rows.head())

# To save the result to a new CSV file (optional)
output_file = "25-11-15_Ref004_Edits.csv"
df.to_csv(output_file, index=False)
print(f"\nSaved results to {output_file}")

/tmp/ipython-input-2179306554.py:7: DtypeWarning: Columns (29,31,32,33,34) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Successfully loaded /content/drive/MyDrive/25-11-07_Ref004_Edits.csv. Total rows: 35424

Building conflict author set (for Condition 4)...
Number of unique authors satisfying Condition 1 (short core, empty role): 2110
Number of unique authors satisfying Condition 2 (short core, multiple non-empty roles): 2187

Found 3261 unique conflict authors based on both criteria.

First 30 conflict authors:
['赵宗', '谢森', '柏立', '夏迁', '沪生', '查烈', '晨钟', '张炎', '黄伞', '红铁', '茹桂', '美良', '舒华', '田川', '祖慰', '忻趵', '宋巍', '孙镛', '辽莎', '任伍', '傅丹', '赵勋', '广海', '沙洁', '若水', '史一', '沪民', '唐人', '纪华', '云皆']

Applying flagging logic to all rows... (this may take a moment)
Flagging complete.

--- Flag Summary ---
Flag
N          27928
Y=1         4381
Y=2         1578
Y=3          693
Y=1,2        331
Y=1,3        227
Y=4          137
Y=2,3         45
Y=2,4         31
Y=5           27
Y=1,4         19
Y=1,2,3        9
Y=3,4          4
Y=3,5          4
Y=1,2,4        3
Y=6            2
Y=2,5          2
Y=1,5          2
Y=1